# 01 · Data Cleaning

This notebook inspects the original IBM Telco Customer Churn dataset, documents data quality checks, applies justified corrections, and saves a separate cleaned CSV. The original dataset file is not modified.

## 1. Load and inspect

The target is `Churn`. `customerID` is retained for reporting but will not be used as a model feature.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Datasets').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'Customer_Churn_Prediction'
DATA_PATH = PROJECT_ROOT / 'Datasets' / 'Telco-Customer-Churn.csv'
CLEANED_PATH = PROJECT_ROOT / 'Datasets' / 'Telco-Customer-Churn_Cleaned.csv'

raw = pd.read_csv(DATA_PATH)
df = raw.copy()
print('Shape:', df.shape)
display(df.head())
display(df.dtypes.rename('dtype').to_frame())

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


,dtype
customerID,str
gender,str
SeniorCitizen,int64
Partner,str
Dependents,str
tenure,int64
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


## 2. Data quality checks

Blank strings are checked separately from null values because `TotalCharges` is stored as text in the original file. Numeric outliers are screened with the 1.5 × IQR rule; this is a review flag only, not a reason to delete records.

In [2]:
profile = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'null_values': raw.isna().sum(),
    'blank_strings': raw.astype(str).apply(lambda column: column.str.strip().eq('').sum()),
    'unique_values': raw.nunique(dropna=True),
})
display(profile)
print('Duplicate full rows:', raw.duplicated().sum())
print('Churn distribution:')
display(raw['Churn'].value_counts(dropna=False).to_frame('customers'))
print('Categorical values:')
for column in raw.select_dtypes(include=['object', 'str']).columns:
    if column != 'customerID' and column != 'TotalCharges':
        print(column, sorted(raw[column].dropna().unique().tolist()))

,dtype,null_values,blank_strings,unique_values
customerID,str,0,0,7043
gender,str,0,0,2
SeniorCitizen,int64,0,0,2
Partner,str,0,0,2
Dependents,str,0,0,2
tenure,int64,0,0,73
PhoneService,str,0,0,2
MultipleLines,str,0,0,3
InternetService,str,0,0,3
OnlineSecurity,str,0,0,3


Duplicate full rows: 0
Churn distribution:


,customers
Churn,
No,5174
Yes,1869


Categorical values:
gender ['Female', 'Male']
Partner ['No', 'Yes']
Dependents ['No', 'Yes']
PhoneService ['No', 'Yes']
MultipleLines ['No', 'No phone service', 'Yes']
InternetService ['DSL', 'Fiber optic', 'No']
OnlineSecurity ['No', 'No internet service', 'Yes']
OnlineBackup ['No', 'No internet service', 'Yes']
DeviceProtection ['No', 'No internet service', 'Yes']
TechSupport ['No', 'No internet service', 'Yes']
StreamingTV ['No', 'No internet service', 'Yes']
StreamingMovies ['No', 'No internet service', 'Yes']
Contract ['Month-to-month', 'One year', 'Two year']
PaperlessBilling ['No', 'Yes']
PaymentMethod ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
Churn ['No', 'Yes']


In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
blank_total_charges = df['TotalCharges'].isna()
zero_tenure_blank = blank_total_charges & df['tenure'].eq(0)
print('TotalCharges blanks:', int(blank_total_charges.sum()))
print('Blanks with zero tenure:', int(zero_tenure_blank.sum()))
print('Blanks with nonzero tenure:', int((blank_total_charges & df['tenure'].ne(0)).sum()))
df.loc[zero_tenure_blank, 'TotalCharges'] = 0.0
if df['TotalCharges'].isna().any():
    df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['SeniorCitizen'] = df['SeniorCitizen'].astype('category')

continuous_columns = ['tenure', 'MonthlyCharges', 'TotalCharges']
outlier_rows = []
for column in continuous_columns:
    lower_quartile = df[column].quantile(0.25)
    upper_quartile = df[column].quantile(0.75)
    spread = upper_quartile - lower_quartile
    count = int(((df[column] < lower_quartile - 1.5 * spread) | (df[column] > upper_quartile + 1.5 * spread)).sum())
    outlier_rows.append({'column': column, 'iqr_outlier_count': count})
display(pd.DataFrame(outlier_rows))
print('Rows retained:', len(df))

TotalCharges blanks: 11
Blanks with zero tenure: 11
Blanks with nonzero tenure: 0


,column,iqr_outlier_count
0,tenure,0
1,MonthlyCharges,0
2,TotalCharges,0


Rows retained: 7043


## 3. Validate and save

No records are removed. The 11 blank `TotalCharges` values belong to customers with zero tenure and are set to zero. Identifiers remain in the cleaned file for traceability but are excluded during model training. Median and most-frequent imputers remain inside the model pipelines to handle any future missing values without fitting preprocessing on held-out data.

In [4]:
expected_categories = {
    'gender': {'Female', 'Male'},
    'Partner': {'Yes', 'No'},
    'Dependents': {'Yes', 'No'},
    'PhoneService': {'Yes', 'No'},
    'MultipleLines': {'Yes', 'No', 'No phone service'},
    'InternetService': {'DSL', 'Fiber optic', 'No'},
    'OnlineSecurity': {'Yes', 'No', 'No internet service'},
    'OnlineBackup': {'Yes', 'No', 'No internet service'},
    'DeviceProtection': {'Yes', 'No', 'No internet service'},
    'TechSupport': {'Yes', 'No', 'No internet service'},
    'StreamingTV': {'Yes', 'No', 'No internet service'},
    'StreamingMovies': {'Yes', 'No', 'No internet service'},
    'Contract': {'Month-to-month', 'One year', 'Two year'},
    'PaperlessBilling': {'Yes', 'No'},
    'PaymentMethod': {'Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'},
    'Churn': {'Yes', 'No'},
}
category_check = pd.DataFrame([
    {'column': column, 'observed_values': sorted(df[column].astype(str).unique().tolist()), 'unexpected_values': sorted(set(df[column].astype(str).unique()) - allowed)}
    for column, allowed in expected_categories.items()
])
display(category_check)
df.to_csv(CLEANED_PATH, index=False)
print('Cleaned dataset saved:', CLEANED_PATH)
print('Cleaned shape:', df.shape)
print('Remaining missing values:', int(df.isna().sum().sum()))

,column,observed_values,unexpected_values
0,gender,"[Female, Male]",[]
1,Partner,"[No, Yes]",[]
2,Dependents,"[No, Yes]",[]
3,PhoneService,"[No, Yes]",[]
4,MultipleLines,"[No, No phone service, Yes]",[]
5,InternetService,"[DSL, Fiber optic, No]",[]
6,OnlineSecurity,"[No, No internet service, Yes]",[]
7,OnlineBackup,"[No, No internet service, Yes]",[]
8,DeviceProtection,"[No, No internet service, Yes]",[]
9,TechSupport,"[No, No internet service, Yes]",[]


Cleaned dataset saved: D:\Documents\ChatGPT\predictive\Customer_Churn_Prediction\Datasets\Telco-Customer-Churn_Cleaned.csv
Cleaned shape: (7043, 21)
Remaining missing values: 0
